# iprPy melting_temperature calculation

In [1]:
# Standard library imports
import datetime
from copy import deepcopy

# http://www.numpy.org/
import numpy as np

# https://ipython.org/
from IPython.display import display, Code, Markdown, Pretty

# https://github.com/usnistgov/atomman
import atomman as am
import atomman.unitconvert as uc

# https://github.com/usnistgov/iprPy
import iprPy

print('Notebook last executed on', datetime.date.today(), 'using iprPy version', iprPy.__version__)

Notebook last executed on 2026-06-30 using iprPy version 0.12.a


## 1. Load calculation and view description

### 1.1. Load the calculation

In [2]:
# Load the calculation being demoed
calculation = iprPy.load_calculation('melting_temperature')

### 1.2. Display calculation description and theory

In [3]:
# Display main docs and theory
display(Markdown(calculation.maindoc))
display(Markdown(calculation.theorydoc))

# melting_temperature calculation style

**Lucas M. Hale**, [lucas.hale@nist.gov](mailto:lucas.hale@nist.gov?Subject=ipr-demo), *Materials Science and Engineering Division, NIST*.

## Introduction

The melting_temperature calculation style attempts to determine the melting temperature for an element by constructing a two-phase system consisting of half solid and half liquid, then relaxing the system with an nph simulation.  If the initial guess temperature is close to the melting temperature, then it is expected that the system will equilibrate towards the melting temperature as one phase transforms into the other.  Good direct estimates for the melting temperature are obtained when the final configuration retains substantial amounts of both phases.

### Version notes

- v0.11.4: calculation method added.
- v0.12.0: Method updated to support the LAMMPS library interface.
  
### Additional dependencies

### Disclaimers

- [NIST disclaimers](http://www.nist.gov/public_affairs/disclaimer.cfm)

- Good estimates require the existence of both phases at the end of the simulation in substantial amounts.  This typically requires iterating the simulation multiple times where the input guess temperature is updated until the fraction of both phases falls within some target range.  Note that there is only a loose correspondence between the guess temperature and the equilibrium temperature, so iterations need to be based on phase amounts not output temperatures!

- The calculation method uses polyhedral template matching (ptm) to provide an automatic estimate of the solid phase amount.  The ptm method requires specifying which crystal structure(s) are expected and estimates are based on that.  For crystal structures not currently supported by ptm, you will need to estimate the phases yourself using the generated dump files.

- The melting temperature is sensitive to the choice of the solid phase and multiple estimates can be possible if multiple solid phases are (meta)stable at high temperatures.

- This method is designed for estimating the melting temperature for single element systems.

## Method and Theory

First, an initial system is constructed by creating a supercell of a solid crystal phase.  The structure should be roughly the same dimensions along the a- and b-axes, and roughly twice that along the c-axis.  Two regions, a top and bottom, are defined such that they both encompass half of the system divided along the middle of the z-axis.

The system is iterated using an nph barostat for the target pressure, plus Berendsen thermostats set at the initial liquid temperature (T_liquid) value for the top region and set at the initial solid temperature (T_solid) value for the bottom region.  The aim of this simulation stage is to melt the crystal in the top region while keeping the crystal in the bottom region from transforming.  After an initial melt period, the thermostats are updated to gradually migrate the temperatures of the two regions towards the guess temperature (T_guess).  Finally, the thermostats are removed and the system is allowed to continue relaxing with only the barostat.

The default behavior of the calculation sets T_liquid = 1.25 T_guess, and T_solid = 0.5 T_guess thereby reducing the number of required input temperature values to one.  This seems to be a good choice for most cases, but you can freely define all three temperatures separately if you wish.

The resulting melting temperature is estimated as the mean of the temperature for the second half of the final nph run.  If reference crystal structures were specified for use with the polyhedral template matching method then the fraction of solid elements is estimated for each dump file generated in the second half of the final nph run.

__NOTE__: The nature of this calculation is best suited as a unit of work to be integrated into an iterative workflow that tries different guess temperatures until the simulation results in solid fractions within some target range around 0.50.  Good melting temperature estimates can be estimated by averaging the measured melting temperature for multiple runs of this calculation that all result in acceptable solid fractions.  See the bin/melt_commander directory for one such implementation.

### 2.1. Show code and supporting file names and content

In [4]:
# Display calculation code and supporting files
for filename, contents in calculation.files.items():
    display(Markdown(f'## Contents of file "{filename}"'))
    if filename[-3:] == '.py':
        display(Code(contents, language='python'))
    else:
        display(Pretty(contents))

## Contents of file "melting_temperature.py"

# Python script created by Lucas Hale

# Standard library imports
from typing import Optional, Union
from pathlib import Path
import random

# http://www.numpy.org/
import numpy as np

# https://github.com/usnistgov/atomman 
import atomman as am
import atomman.unitconvert as uc
from atomman.typing import lammpspotential, unitfloat
from atomman.lammps import LAMMPS, LAMMPSobj

def melting_temperature(lammps_command: Union[str, LAMMPSobj],
                        system: am.System,
                        potential: lammpspotential,
                        temperature_guess: float,
                        mpi_command: Optional[str] = None,
                        pressure: unitfloat = 0.0,
                        temperature_solid: Optional[float] = None,
                        temperature_liquid: Optional[float] = None,
                        ptm_structures: Optional[str] = None,
                        meltsteps: int = 10000,
                        scalesteps: int = 10000,
                        runsteps: int = 200000,
                        thermosteps: int = 100,
                        dumpsteps: Optional[int] = None,
                        randomseed1: Optional[int] = None,
                        randomseed2: Optional[int] = None,
                        usefiles: bool = False) -> dict:
    """
    Creates a solid-liquid phase coexistence simulation to estimate the melting
    temperature.  The boundary for the two phases will be perpendicular to the
    z-axis and positioned halfway along the c box vector.
    
    Parameters
    ----------
    lammps_command : str, LAMMPSEXE or LAMMPSLIB
        LAMMPS executable command, LAMMPS library name, or an atomman LAMMPS
        interface object.
    system : atomman.System
        The initial system to perform the calculation on.  This should be a
        supercell with dimensions along the z direction roughly twice the
        dimensions in the other directions.
    potential : PotentialLAMMPS or PotentialLAMMPSKIM
        The LAMMPS implemented potential to use.
    temperature_guess : float, optional
        The initial guess for the melting temperature. The closer to the actual
        temperature the faster and more likely convergence will be possible.
    mpi_command : str, optional
        The MPI command for running LAMMPS in parallel.  If not given, LAMMPS
        will run serially.
    pressure : float or str, optional
        The target pressure to use with the barostat.  Default value is 0.0.
    temperature_liquid : float or None, optional
        The initial temperature to use for the liquid region to melt the crystal.
        Default value of None will use 1.25 * temperature_guess.
    temperature_solid : float or None, optional
        The initial temperature to use for the solid region.
        Default value of None will use 0.5 * temperature_guess.
    meltsteps : int, optional
        The number of integration steps to perform with half of the system
        at temperature_solid and half of the system at temperature_liquid to
        create the two phase configuration.  Default value is 10000.
    scalesteps : int, optional
        The number of integration steps after meltsteps where the temperature
        of the atoms in the two phases are both scaled to temperature_guess.
        This ensures that the full system starts near temperature_guess for the
        main runsteps.  Default value is 10000.
    runsteps : int, optional
        The number of nph integration steps to perform on the two-phase system
        to hopefully get a stable coexistence at the melting temperature.
    thermosteps : int, optional
        Thermo values will be reported every this many steps. Default is
        100.
    dumpsteps : int or None, optional
        Dump files will be saved every this many steps. Default is None,
        which sets dumpsteps equal to meltsteps + scalesteps + runsteps.
    randomseed1 : int or None, optional
        Random number seed used by LA

### 2.2. Optional: Save supporting files

Set "savefiles = True" to save files locally.  

Note that the code above should be using atomman.tools.read_calc_file() to read the files, which will read any local files with matching names if they exist or read the packaged version if the local files do not exist. This means that if you save the files locally, you can modify them and see how it affects the calculation!

In [5]:
savefiles = False

if savefiles:
    for filename, contents in calculation.files.items():
        if filename[-3:] != '.py':
            with open(filename, 'w') as f:
                f.write(contents)

## 3. Specify input parameters

### 3.1. System-specific paths

- __lammps_command__ is the LAMMPS command to use (required).
- __mpi_command__ MPI command for running LAMMPS in parallel. A value of None will run simulations serially.

In [6]:
lammps_command = 'F:/LAMMPS/current/bin/lmp.exe'
mpi_command = None
#mpi_command = 'mpiexec -localonly 4'

# Optional: check that LAMMPS works and show its version 
print(f'LAMMPS version = {am.lammps.checkversion(lammps_command)["version"]}')

LAMMPS version = 23 Jun 2022 - Update 2


### 3.2. Interatomic potential

- __potential_name__ gives the name of a potential_LAMMPS record to find and download from the iprPy library.  
- __potential__ is a potential_LAMMPS or potential_LAMMPS_KIM record object (required).

See documentation for the [potentials package](https://github.com/usnistgov/potentials/tree/master/doc) for more options on finding, loading and building potential objects (doc Notebook #s 0, 5.3, 5.4 and 7).

In [7]:
potential_name = '1999--Mishin-Y--Ni--LAMMPS--ipr1'

# Retrieve potential and parameter file(s) using atomman
potential = am.load_lammps_potential(id=potential_name, getfiles=True)

### 3.3. Initial unit cell system

- __ucell__ is an atomman.System representing a fundamental unit cell of the system (required).  Here, this is generated using a crystal prototype, lattice constants and symbols.

See documentation for the [atomman package](https://github.com/lmhale99/atomman/tree/master/doc/tutorial) for more options on building and loading atomic configurations (doc Notebook #s 1.1, 1.2, 1.3, 1.4 and 1.4.*)

In [8]:
# Create ucell by loading prototype record
ucell = am.load('prototype', 'A1--Cu--fcc', symbols='Ni', a=3.6)

print(ucell)

avect =  [ 3.600,  0.000,  0.000]
bvect =  [ 0.000,  3.600,  0.000]
cvect =  [ 0.000,  0.000,  3.600]
origin = [ 0.000,  0.000,  0.000]
natoms = 4
natypes = 1
symbols = ('Ni',)
pbc = [ True  True  True]
per-atom properties = ['atype', 'pos']
     id |   atype |  pos[0] |  pos[1] |  pos[2]
      0 |       1 |   0.000 |   0.000 |   0.000
      1 |       1 |   0.000 |   1.800 |   1.800
      2 |       1 |   1.800 |   0.000 |   1.800
      3 |       1 |   1.800 |   1.800 |   0.000


### 3.4. System modifications

- __sizemults__ list of three integers specifying how many times the ucell vectors of $a$, $b$ and $c$ are replicated in creating system.  NOTE that for this calculation the final $c$ dimension should be roughly twice that of the other two dimensions.
- __system__ is an atomman.System to perform the scan on (required). 

In [9]:
sizemults = [10, 10, 20]

# Generate system by supersizing ucell
system = ucell.supersize(*sizemults)
print('# of atoms in system =', system.natoms)

# of atoms in system = 8000


### 3.5. Calculation-specific parameters

- __temperature_guess__ is an input temperature to use for setting up the liquid and solid regions. Good values are likely close to the actual melting temperature, but direct correspondence is not guaranteed.
- __pressure__ is the target pressure to use with the barostat.  Default value is 0.0.
- __temperature_liquid__ is the initial temperature to use for the liquid region to melt the crystal. Default value of None will use 1.25 * temperature_guess.
- __temperature_solid__ is the initial temperature to use for the solid region. Default value of None will use 0.5 * temperature_guess.
- __meltsteps__ is the number of integration steps to perform with half of the system at temperature_solid and half of the system at temperature_liquid to create the two phase configuration.  Default value is 10000.
- __scalesteps__ is the number of integration steps after meltsteps where the temperature of the atoms in the two phases are both scaled to temperature_guess. This ensures that the full system starts near temperature_guess for the main runsteps.  Default value is 10000.
- __runsteps__ is the number of nph integration steps to perform on the two-phase system to hopefully get a stable coexistence at the melting temperature.
- __thermosteps__ indicates how often thermo values will be reported. Default is 100.
- __dumpsteps__ indicates how often to save dump files. Default is None, which sets dumpsteps equal to meltsteps + scalesteps + runsteps.
- __randomseed1__ is a random number seed used by LAMMPS in creating velocities for the liquid region.  Default is None which will select random ints between 1 and 900000000.
- __randomseed2__ is a random number seed used by LAMMPS in creating velocities for the solid region.  Default is None which will select random ints between 1 and 900000000.

In [10]:
temperature_guess = 2200
pressure = 0.0
temperature_solid = None
temperature_liquid = None
ptm_structures = 'fcc'
meltsteps = 10000
scalesteps = 10000
runsteps = 200000
thermosteps = 100
dumpsteps = 10000
randomseed1 = None
randomseed2 = None

## 4. Run calculation and view results

### 4.1. Run calculation

All primary calculation method functions take a series of inputs and return a dictionary of outputs.

In [11]:
# What is calculation.calc an alias of?
calculation.calc.__module__

'iprPy.calculation.melting_temperature.melting_temperature'

In [12]:
results_dict = calculation.calc(lammps_command, system, potential, temperature_guess,
                                mpi_command = mpi_command,
                                pressure = pressure,
                                temperature_solid = temperature_solid,
                                temperature_liquid = temperature_liquid,
                                ptm_structures = ptm_structures,
                                meltsteps = meltsteps,
                                scalesteps = scalesteps,
                                runsteps = runsteps,
                                thermosteps = thermosteps,
                                dumpsteps = dumpsteps,
                                randomseed1 = randomseed1,
                                randomseed2 = randomseed2)
print(results_dict.keys())

dict_keys(['melting_temperature', 'fraction_solids'])


### 4.2. Report results

Values returned in the results_dict:

- **'melting_temperature'** (*float*) - The equilibrium temperature taken from the second half of the runsteps.
- **'fraction_solids'** (*list*) - Polyhedral template matching estimates of the fraction of the system that is of the expected solid phase, taken for each generated dump file.

In [15]:
print(np.array(results_dict['fraction_solids']))

if results_dict['fraction_solids'][-1] < 0.2:
    print('Warning! Too little solid in final configuration! Maybe try a lower guess temperature')
elif results_dict['fraction_solids'][-1] > 0.8:
    print('Warning! Too little liquid in final configuration! Maybe try a higher guess temperature')
else:
    print(f'Tm = {results_dict['melting_temperature']}')


[0.54975  0.54875  0.542375 0.5575   0.52575  0.537375 0.5085   0.524125
 0.53375  0.5285   0.550625]
Tm = 2199.6950041797654


### 4.3. Optional: Clean calculation files

The calculation may generate output files when it runs.  Calling calculation.clean_files() will delete any generated files to keep the workspace clean.

In [14]:
calculation.clean_files()